# Segmentation

The [Quickstart](quickstart.ipynb) reduced data one way: fewer **periods** (a year
of days → a handful of typical days). **Segmentation** is the second, independent
lever — it reduces the resolution *within* each period.

Instead of keeping all 24 hourly steps of a typical day, segmentation merges
adjacent steps into a few **variable-length segments**: long where the profile is
flat, short where it changes quickly. The two levers multiply, so they are the
two halves of your time-step budget.

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

## One period, fewer steps

Take the same 6 typical days as the Quickstart, but describe each one with **6
segments** instead of 24 hours:

In [ ]:
from tsam import SegmentConfig

result = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    segments=SegmentConfig(n_segments=6),
)

The bold line is the segmented typical day; the faint lines are the real days it
stands for. Notice the typical day is now a **step function** — each step is one
segment, held flat across the hours it covers:

In [ ]:
result.plot.cluster_members(columns=["Load"])

## Segments are not equal width

tsam chooses where the segment boundaries fall, so segments absorb more hours
where the signal is steady and fewer where it moves. Averaged across the typical
days:

In [ ]:
result.plot.segment_durations()

## The budget: periods × segments

Without segmentation, 6 typical days cost 6 × 24 = 144 time steps. With 6 segments
each, that drops to 6 × 6 = 36 — a quarter of the steps. The accuracy cost is read
the same way as always:

In [ ]:
plain = tsam.aggregate(data, n_clusters=6, period_duration="1D")
pd.DataFrame(
    {
        "6 days x 24 h": plain.accuracy.rmse,
        "6 days x 6 seg": result.accuracy.rmse,
    }
).round(4)

So you have two dials: how many typical periods, and how fine each one is. Picking
the best combination by hand is tedious — [Sizing & tuning](sizing_tuning.ipynb)
searches both for you against a target.